# Resume train YOLOv8n — thêm 50 epochs (epoch 51 → 100)

Notebook này **train tiếp** từ checkpoint của run trước, thay vì train lại từ đầu.

**TRƯỚC KHI CHẠY:**
1. Settings ⚙ bên phải → Accelerator: **GPU T4 x2** → Internet **ON**.
2. **Add-ons → Secrets → ADD A NEW SECRET**: Label `ROBOFLOW_API_KEY`, Value = key Roboflow của bạn, bật Attach to notebook. (Chỉ làm 1 lần.)
3. Nếu run trước đã lưu `last.pt`/`best.pt` trong **Output**, tải về rồi **Add input → Dataset (private)**.
4. Chạy lần lượt từng ô (hoặc Save & Run All).

Model tự nhận diện:
- Có **`last.pt` + `args.yaml`** → `resume=True`, chạy tiếp từ epoch ~50 tới 100.
- Chỉ có **`best.pt`** → load làm trọng số khởi tạo, train tiếp 50 epochs (fine-tune).
- Không tìm thấy file nào → dừng và nhắc bạn upload.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics roboflow

from ultralytics import YOLO
import torch, os, glob, shutil, yaml
print("Ultralytics:", __import__("ultralytics").__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

In [ ]:
# Tải dataset từ Roboflow.
# Lấy key từ Kaggle Secret (đã thêm 1 lần qua UI). Fallback sang env var / .env nếu chạy local.
import os
_key = os.environ.get("ROBOFLOW_API_KEY", "")
if not _key:
    try:
        from kaggle_secrets import UserSecretsClient
        _key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
    except Exception:
        pass
if not _key:
    raise ValueError("Thiếu ROBOFLOW_API_KEY. Trên Kaggle: Add-ons → Secrets → tạo secret tên ROBOFLOW_API_KEY.")
os.environ["ROBOFLOW_API_KEY"] = _key

from roboflow import Roboflow
WORKSPACE       = "trantungbach26-gmail-com"
PROJECT_NAME    = "citrus-disease-detection-yoydc-ahtka"
PROJECT_VERSION = 1
rf = Roboflow(api_key=_key)
project = rf.workspace(WORKSPACE).project(PROJECT_NAME)
project.version(PROJECT_VERSION).download("yolov8")

In [ ]:
cands = glob.glob("/kaggle/working/*/data.yaml")
DATASET_PATH = os.path.dirname(cands[0]) if cands else "/kaggle/working/citrus-disease-detection-1"
print("DATASET_PATH =", DATASET_PATH)

cfg = yaml.safe_load(open(os.path.join(DATASET_PATH, "data.yaml")))
print("nc:", cfg["nc"])
# Vá path tuyệt đối (data.yaml Roboflow dùng ../ không đúng trên Kaggle)
for key, sub in (("train", "train/images"), ("val", "val/images")):
    p = os.path.join(DATASET_PATH, sub)
    if os.path.isdir(p):
        cfg[key] = p
yaml.safe_dump(cfg, open(os.path.join(DATASET_PATH, "data.yaml"), "w"))

In [ ]:
# ===== TÌM TRỌNG SỐ RUN TRƯỚC =====
# Ưu tiên last.pt (cho phép resume nối tiếp). Nếu chỉ có best.pt thì vẫn dùng được.
import glob as _glob

# In ra để debug — biết file nào thực sự có trong input
print("=== Files trong /kaggle/input ===")
for f in _glob.glob("/kaggle/input/**/*", recursive=True):
    print(" ", f)

last_cands = _glob.glob("/kaggle/input/**/last.pt", recursive=True)            + _glob.glob("/kaggle/working/**/last.pt", recursive=True)
best_cands = _glob.glob("/kaggle/input/**/best.pt", recursive=True)            + _glob.glob("/kaggle/working/**/best.pt", recursive=True)            + _glob.glob("/kaggle/input/**/*checkpoint*.pt", recursive=True)

last_pt = last_cands[0] if last_cands else None
best_pt = best_cands[0] if best_cands else None

# resume=True cần last.pt đi kèm args.yaml cùng thư mục
args_near = None
if last_pt:
    args_near = os.path.join(os.path.dirname(last_pt), "args.yaml")
    if not os.path.exists(args_near):
        args_near = None

WEIGHTS = last_pt if (last_pt and args_near) else (best_pt or last_pt)

if not WEIGHTS:
    raise FileNotFoundError("Không tìm thấy last.pt/best.pt. Tải từ Output run trước rồi Add input (private) vào kernel này.")

print("Dùng trọng số:", WEIGHTS)
print("resume nối tiếp:", bool(last_pt and args_near))

In [ ]:
EPOCHS   = 100 if (last_pt and args_near) else 50   # 100+resume=True => chạy 51->100; không resume thì 50 mớiIMGSZ    = 640BATCH    = 16PATIENCE = 15OUT_DIR = "/kaggle/working/drone_yolo"os.makedirs(OUT_DIR, exist_ok=True)# Auto-backup best.pt sau mỗi epochdef _backup(trainer):    try:        src = os.path.join(trainer.save_dir, "weights", "best.pt")        shutil.copy(src, os.path.join(OUT_DIR, "best_checkpoint.pt"))        print(f"  [backup epoch {trainer.epoch}] -> {OUT_DIR}", flush=True)    except Exception as e:        print("  [backup fail]", e, flush=True)from ultralytics.utils import callbackscallbacks.default_callbacks["on_fit_epoch_end"].append(_backup)# ===== TĂNG RECALL (model hiện P~0.66 / R~0.45) =====# - class_weights=True: lớp ít ảnh nặng hơn -> bắt được nhiều hơn# - fliplr 0.5 + scale 0.5: đa dạng vị trí/kích thước lá# - cos_lr để lr giảm mượt về cuối# Giữ nguyên các tăng cường biến dạng hình dạng (degrees/shear/perspective) vì# lá bệnh có hình dạng có ý nghĩa — biến dạng quá làm sai nhãn.EXTRA = dict(    class_weights=True,          # cân bằng lớp hiếm    fliplr=0.5, scale=0.5,       # lật ngang + zoom ngẫu nhiên    cos_lr=True,                 # learning rate giảm mượt    seed=42,)model = YOLO(WEIGHTS)train_kwargs = dict(    data=f"{DATASET_PATH}/data.yaml",    epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,    patience=PATIENCE, device=0,    cache=True, workers=2,    project="/kaggle/working/runs", name="drone_yolov8n",)train_kwargs.update(EXTRA)if last_pt and args_near:    train_kwargs["resume"] = Trueresults = model.train(**train_kwargs)

In [ ]:
# Val + in cả recall — quan trọng cho model 'chính xác nhưng bỏ sót nhiều'metrics = model.val()print(f"precision: {metrics.box.p:.4f}  recall: {metrics.box.r:.4f}")print(f"mAP50:     {metrics.box.map50:.4f}")print(f"mAP50-95:  {metrics.box.map:.4f}")print("Bảng per-class (đếm class recall thấp -> cần thêm dữ liệu hoặc augmentation):")m = metrics.boxfor i, name in enumerate(model.names.values()):    ap = m.ap[i, 0]  # AP50 của class i    if ap >= 0:        print(f"  {name:28s} AP50={ap:.3f}")

In [ ]:
RESULTS_DIR = "/kaggle/working/runs/drone_yolov8n/weights"
model.export(format="onnx", imgsz=640, opset=11, simplify=True)
for f in glob.glob(RESULTS_DIR + "/best.*"):
    shutil.copy(f, os.path.join(OUT_DIR, os.path.basename(f)))
    print("Đã copy output:", os.path.basename(f))
print(">>> Tải: panel Output bên phải → Download.")

## Nếu bạn chưa upload được trọng số cũ

Run trước lỡ tắt session mà chưa lưu `last.pt` ra Output:

- Vào kernel cũ → tab **Output** → tải `best.zip` (hoặc `last.zip`).
- Ở kernel này: **Add input → Dataset (New)** → tải zip lên (private).
- Chạy lại ô tìm trọng số. Nếu chỉ có `best.pt`, model vẫn train tiếp 50 epochs fine-tune (kết quả vẫn tốt).

> Key Roboflow lấy từ Kaggle Secret — không hardcode, không lộ ra ngoài.